# DocMind RAG — Baseline Evaluation
FinanceBench dataset. Default config: Parent-Child (800/150), text-embedding-3-small, GPT-4o.

In [ ]:
import json
import httpx

API_BASE = "http://localhost:8000/api/v1"

# Start eval run
response = httpx.post(f"{API_BASE}/eval/run", json={
    "dataset": "financebench",
    "sample_size": 50,
})
result = response.json()
print(f"Eval started: {result}")
run_id = result.get("run_id") or result.get("task_id", "")

In [ ]:
# Poll for results (run multiple times)
import time
for _ in range(30):
    r = httpx.get(f"{API_BASE}/eval/results/{run_id}")
    data = r.json()
    if data.get("status") in ("completed", "failed"):
        break
    print(f"Status: {data.get('status')}...")
    time.sleep(10)

print(json.dumps(data, indent=2))

In [ ]:
# Save results
if data.get("status") == "completed":
    with open("../results/financebench_results.json", "w") as f:
        json.dump(data, f, indent=2)
    print("Results saved!")

    metrics = data.get("metrics", {})
    print(f"\n{'='*40}")
    print(f"Retrieval Hit Rate: {metrics.get('retrieval_hit_rate', 0):.1%}")
    print(f"Faithfulness:       {metrics.get('faithfulness', 0):.4f}")
    print(f"Answer Relevancy:   {metrics.get('answer_relevancy', 0):.4f}")
    print(f"Context Recall:     {metrics.get('context_recall', 0):.4f}")
    print(f"Latency p95:        {metrics.get('latency_p95_ms', 0):.0f}ms")
    print(f"{'='*40}")